<a href="https://colab.research.google.com/github/shravan1808/DeepLearningProjects/blob/main/Project_05/Project_05_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class PretrainedBackbone(nn.Module):
  def __init__(self):
    super().__init__()
    self.block1 = nn.Sequential(
        nn.Linear(512,256),
        nn.BatchNorm1d(256),
        nn.ReLU()
    )
    self.block2  = nn.Sequential(
        nn.Linear(256,128),
        nn.BatchNorm1d(128),
        nn.ReLU()
    )

  def forward(self, x):
    x = self.block1(x)
    x = self.block2(x)
    return x

  def freeze_backbone(self):
    for param in self.parameters():
      param.requires_grad = False

  def unfreeze_upper_layers(self):
    for param in self.block2.parameters():
      param.requires_grad = True
    for param in self.block1.parameters():
      param.requires_grad = False

In [ ]:
class TransferLearningClassifier(nn.Module):
  def __init__(self,backbone):
    super().__init__()
    self.backbone = backbone
    self.classifier_head = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,3)
    )

  def forward(self,x):
    x = self.backbone(x)
    x = self.classifier_head(x)
    return x


In [ ]:
backbone = PretrainedBackbone()
model = TransferLearningClassifier(backbone)

In [ ]:
torch.manual_seed(42)

X = torch.randn(300, 512)
y = torch.randint(0, 3, (300,))

In [ ]:
from torch.utils.data import Dataset,DataLoader

class FeatureDataset(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self,idx):
    return self.X[idx], self.y[idx]

In [ ]:
dataset = FeatureDataset(X,y)

train_loader = DataLoader(dataset,batch_size=32,shuffle=True)

model.backbone.freeze_backbone()

optimizer = optim.Adam(
    model.classifier_head.parameters(),
    lr = 0.01
)

loss_fn = nn.CrossEntropyLoss()

phase1_loss_history = []
for epoch in range(10):
  epoch_loss = 0
  for X_batch,y_batch in train_loader:
    optimizer.zero_grad()
    output = model(X_batch)
    loss = loss_fn(output,y_batch)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()
  avg_loss = epoch_loss / len(train_loader)
  phase1_loss_history.append(avg_loss)

In [ ]:
model.backbone.unfreeze_upper_layers()

In [ ]:
optimizer_ph2 = optim.Adam([
    {'params':model.backbone.block2.parameters(),'lr':0.0001},
    {'params':model.classifier_head.parameters(),'lr':0.001}
])

phase2_loss_history = []
for epoch in range(15):
  epoch_loss = 0
  for X_batch,y_batch in train_loader:
    optimizer_ph2.zero_grad()
    output = model(X_batch)
    loss = loss_fn(output,y_batch)
    loss.backward()
    optimizer_ph2.step()
    epoch_loss += loss.item()
  avg_loss = epoch_loss / len(train_loader)
  phase2_loss_history.append(avg_loss)

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for X_batch,y_batch in train_loader:
    output = model(X_batch)
    predicted = torch.argmax(output,dim=1)
    total += y_batch.size(0)
    correct += (predicted == y_batch).sum().item()

accuracy = correct / total

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'loss': phase2_loss_history[-1],
    'accuracy': accuracy

}
torch.save(checkpoint, 'finetuned_model.pt')
